# UD2.04. Procesamiento de imagen con Azure AI Vision

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Práctica P2.2**

Con el texto, el cuerpo de la petición era JSON. Con la imagen **no**, y ahí está la mitad de los
problemas del primer día.

```
POST {endpoint}/computervision/imageanalysis:analyze
     ?api-version=2024-02-01&features=caption,tags,objects,read
```

| Cómo envías la imagen | `Content-Type` | Cuerpo |
|---|---|---|
| Fichero binario | `application/octet-stream` | Los bytes de la imagen |
| URL pública | `application/json` | `{"url": "https://..."}` |

Enviar bytes declarando `application/json` es la causa del **415** que aparece siempre.

Lo que se pide se elige en el parámetro `features` de la URL, y se pueden pedir varias cosas en
la misma llamada, que es más barato que hacer una llamada por cada una:

| `feature` | Devuelve |
|---|---|
| `caption` | Una frase que describe la imagen |
| `denseCaptions` | Varias descripciones, una por región |
| `tags` | Etiquetas con confianza |
| `objects` | Objetos con su recuadro delimitador |
| `read` | OCR: texto de la imagen con su posición |
| `people` | Personas detectadas con su recuadro |
| `smartCrops` | Recortes sugeridos |

Este cuaderno construye `servicios/vision.py` de la práctica P2.2.

In [ ]:
!pip install -q requests python-dotenv pillow

In [ ]:
import io
import os
import pathlib

import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

CLAVE = os.getenv("AZURE_VISION_KEY") or os.getenv("AZURE_LANGUAGE_KEY")
ENDPOINT = (os.getenv("AZURE_VISION_ENDPOINT")
            or os.getenv("AZURE_LANGUAGE_ENDPOINT") or "").rstrip("/")

print("Clave cargada:", bool(CLAVE))
print("Endpoint:     ", ENDPOINT or "(sin definir)")

if not (CLAVE and ENDPOINT):
    print("\nFalta configuración. Repasa el cuaderno UD2.02 antes de seguir.")

Un **recurso multiservicio** de Azure AI Services cubre Language, Vision y Speech con la misma
clave y el mismo endpoint. Por eso la celda anterior acepta las dos variables: si tienes recursos
separados usa las específicas, y si tienes uno multiservicio bastan las de Language.

## 1. Validar antes de enviar

Cada llamada cuesta. Una imagen que el servicio va a rechazar por formato o por tamaño se puede
detectar en local en milisegundos, sin gastar nada.

Los límites de Image Analysis 4.0, al preparar el curso: JPEG, PNG, GIF, BMP, WEBP o TIFF, menos
de 20 MB, y entre 50 × 50 y 16.000 × 16.000 píxeles. **Compruébalos en la documentación**: son
justo el tipo de dato que cambia entre versiones.

In [ ]:
from PIL import Image

FORMATOS = {"JPEG", "PNG", "GIF", "BMP", "WEBP", "TIFF"}
TAMANO_MAXIMO = 20 * 1024 * 1024
LADO_MINIMO = 50
LADO_MAXIMO = 16000


class ImagenNoValida(ValueError):
    """La imagen no cumple los requisitos del servicio."""


def valida_imagen(datos: bytes) -> dict:
    """Comprueba formato y dimensiones antes de gastar una llamada."""
    if len(datos) > TAMANO_MAXIMO:
        raise ImagenNoValida(
            f"La imagen ocupa {len(datos) / 1e6:.1f} MB y el máximo son 20 MB"
        )
    try:
        imagen = Image.open(io.BytesIO(datos))
        imagen.verify()
        imagen = Image.open(io.BytesIO(datos))     # verify() deja el fichero consumido
    except Exception as error:
        raise ImagenNoValida(f"No se ha podido leer la imagen: {error}") from error

    if imagen.format not in FORMATOS:
        raise ImagenNoValida(f"Formato {imagen.format} no admitido. Admitidos: {sorted(FORMATOS)}")

    ancho, alto = imagen.size
    if min(ancho, alto) < LADO_MINIMO:
        raise ImagenNoValida(f"Imagen demasiado pequeña: {ancho}x{alto}, mínimo 50x50")
    if max(ancho, alto) > LADO_MAXIMO:
        raise ImagenNoValida(f"Imagen demasiado grande: {ancho}x{alto}, máximo 16000")

    return {"formato": imagen.format, "ancho": ancho, "alto": alto, "bytes": len(datos)}

In [ ]:
# Comprobación sin llamar a ningún servicio: creamos dos imágenes en memoria
buf = io.BytesIO()
Image.new("RGB", (400, 300), (200, 210, 220)).save(buf, format="PNG")
correcta = buf.getvalue()

buf = io.BytesIO()
Image.new("RGB", (20, 20), (0, 0, 0)).save(buf, format="PNG")
minuscula = buf.getvalue()

print(valida_imagen(correcta))

for datos, etiqueta in [(minuscula, "imagen de 20x20"), (b"esto no es una imagen", "basura")]:
    try:
        valida_imagen(datos)
    except ImagenNoValida as error:
        print(f"{etiqueta:18} -> rechazada: {error}")

## 2. Una sola función de llamada

Cuatro funciones que repiten la misma petición cambiando `features` es código duplicado. Una
función interna hace la llamada y las demás se apoyan en ella.

Lo importante es que **acepta bytes y URL**, y elige el `Content-Type` correcto en cada caso.

In [ ]:
class ErrorServicio(Exception):
    """Fallo al hablar con el servicio."""


URL_VISION = f"{ENDPOINT}/computervision/imageanalysis:analyze"


def _analiza_imagen(imagen, features, idioma="es", timeout=30):
    """Llamada base a Image Analysis.

    imagen: bytes de la imagen, o una cadena con una URL pública.
    features: lista de características, por ejemplo ["caption", "tags"].
    """
    parametros = {
        "api-version": "2024-02-01",
        "features": ",".join(features),
        "language": idioma,
    }
    cabeceras = {"Ocp-Apim-Subscription-Key": CLAVE or ""}

    if isinstance(imagen, str):
        cabeceras["Content-Type"] = "application/json"
        cuerpo = {"json": {"url": imagen}}
    else:
        valida_imagen(imagen)
        cabeceras["Content-Type"] = "application/octet-stream"
        cuerpo = {"data": imagen}

    try:
        respuesta = requests.post(URL_VISION, headers=cabeceras, params=parametros,
                                  timeout=timeout, **cuerpo)
    except requests.RequestException as error:
        raise ErrorServicio(f"No se pudo contactar: {error}") from error

    if respuesta.status_code == 200:
        return respuesta.json()
    if respuesta.status_code == 415:
        raise ErrorServicio(
            "415: el Content-Type no corresponde al cuerpo enviado. "
            "Bytes van con application/octet-stream; una URL, con application/json."
        )
    if respuesta.status_code in (401, 403):
        raise ErrorServicio(f"Credenciales rechazadas (HTTP {respuesta.status_code})")
    if respuesta.status_code == 429:
        raise ErrorServicio("Límite de peticiones superado")
    raise ErrorServicio(f"HTTP {respuesta.status_code}: {respuesta.text[:300]}")

El truco del `**cuerpo` merece una mirada: según el caso, se le pasa a `requests.post` el
argumento `json=` o el argumento `data=`. Es la traducción exacta de la tabla del principio, y
deja la decisión en un solo sitio.

Fíjate también en que el idioma va como parámetro: las descripciones y las etiquetas se devuelven
en el idioma que pidas, cuando el servicio lo soporta para esa característica.

## 3. Las cuatro funciones del módulo

In [ ]:
def describe(imagen, idioma="es") -> dict:
    """Frase que describe la imagen, con su confianza."""
    datos = _analiza_imagen(imagen, ["caption"], idioma)
    caption = datos.get("captionResult") or {}
    return {"texto": caption.get("text"), "confianza": caption.get("confidence")}


def etiqueta(imagen, minimo=0.5, idioma="es") -> list[dict]:
    """Etiquetas por encima de un umbral de confianza."""
    datos = _analiza_imagen(imagen, ["tags"], idioma)
    valores = (datos.get("tagsResult") or {}).get("values", [])
    return [
        {"etiqueta": t["name"], "confianza": t["confidence"]}
        for t in valores if t["confidence"] >= minimo
    ]


def detecta_objetos(imagen, minimo=0.5, idioma="es") -> list[dict]:
    """Objetos detectados, con su recuadro en píxeles de la imagen original."""
    datos = _analiza_imagen(imagen, ["objects"], idioma)
    valores = (datos.get("objectsResult") or {}).get("values", [])
    salida = []
    for objeto in valores:
        etiquetas = objeto.get("tags") or [{}]
        mejor = max(etiquetas, key=lambda t: t.get("confidence", 0))
        if mejor.get("confidence", 0) < minimo:
            continue
        salida.append({
            "objeto": mejor.get("name"),
            "confianza": mejor.get("confidence"),
            "recuadro": objeto["boundingBox"],     # {"x":, "y":, "w":, "h":}
        })
    return salida


def lee_texto(imagen, idioma="es") -> dict:
    """OCR: líneas de texto con su polígono de posición."""
    datos = _analiza_imagen(imagen, ["read"], idioma)
    bloques = (datos.get("readResult") or {}).get("blocks", [])
    lineas = [
        {"texto": linea["text"], "poligono": linea.get("boundingPolygon")}
        for bloque in bloques for linea in bloque.get("lines", [])
    ]
    return {"lineas": lineas, "texto_completo": "\n".join(l["texto"] for l in lineas)}

Un detalle de `detecta_objetos`: un objeto puede traer varias etiquetas candidatas, y la función
se queda con la más confiada. Devolver la primera de la lista, que es lo que se hace sin mirar,
da resultados peores sin que se note por qué.

## 4. Dibujar los recuadros, que es donde se falla

`boundingBox` viene en **píxeles de la imagen original**. Si en la interfaz muestras la imagen
redimensionada, los recuadros hay que **escalarlos con el mismo factor**.

Es el error clásico de esta parte: los recuadros salen desplazados o fuera de la imagen, y cuesta
ver que la causa no está en el servicio sino en el dibujado.

In [ ]:
from PIL import ImageDraw


def dibuja_recuadros(datos_imagen: bytes, objetos: list[dict], ancho_maximo=600) -> Image.Image:
    """Devuelve la imagen redimensionada con los recuadros ya escalados."""
    imagen = Image.open(io.BytesIO(datos_imagen)).convert("RGB")
    ancho_original, alto_original = imagen.size

    factor = min(1.0, ancho_maximo / ancho_original)
    if factor < 1.0:
        imagen = imagen.resize((round(ancho_original * factor), round(alto_original * factor)))

    lienzo = ImageDraw.Draw(imagen)
    for objeto in objetos:
        r = objeto["recuadro"]
        x0, y0 = r["x"] * factor, r["y"] * factor
        x1, y1 = (r["x"] + r["w"]) * factor, (r["y"] + r["h"]) * factor

        lienzo.rectangle([x0, y0, x1, y1], outline=(220, 40, 40), width=3)
        # La etiqueta con el nombre y la confianza: el color no puede ser el único canal
        texto = f"{objeto['objeto']} {objeto['confianza']:.0%}"
        lienzo.rectangle([x0, y0 - 18, x0 + 8 * len(texto), y0], fill=(220, 40, 40))
        lienzo.text((x0 + 3, y0 - 16), texto, fill=(255, 255, 255))

    return imagen

In [ ]:
# Comprobación del escalado sin gastar llamada: una imagen sintética y recuadros inventados
prueba = Image.new("RGB", (1200, 800), (245, 245, 240))
d = ImageDraw.Draw(prueba)
d.rectangle([100, 120, 400, 500], fill=(120, 160, 200))     # "objeto" azul
d.rectangle([700, 300, 1100, 700], fill=(200, 150, 120))    # "objeto" naranja

buf = io.BytesIO()
prueba.save(buf, format="PNG")
datos_prueba = buf.getvalue()

objetos_simulados = [
    {"objeto": "rectangulo azul", "confianza": 0.93,
     "recuadro": {"x": 100, "y": 120, "w": 300, "h": 380}},
    {"objeto": "rectangulo naranja", "confianza": 0.81,
     "recuadro": {"x": 700, "y": 300, "w": 400, "h": 400}},
]

resultado = dibuja_recuadros(datos_prueba, objetos_simulados, ancho_maximo=600)
print("Original:", prueba.size, "-> mostrada:", resultado.size)
resultado

Los recuadros caen exactamente sobre los dos rectángulos, y la imagen se muestra a la mitad de
tamaño. Si quitas el `* factor` de las cuatro coordenadas y vuelves a ejecutar la celda, verás el
error clásico: los recuadros se van fuera.

Merece la pena hacerlo una vez para reconocerlo cuando pase con una imagen de verdad.

## 5. Con imágenes reales

Ahora sí, con el servicio. Usa **imágenes propias o de bancos de dominio público**. No subas
fotografías de personas identificables: la imagen de una persona es un dato personal, y el
tratamiento biométrico tiene protección reforzada.

In [ ]:
RUTA = pathlib.Path("imagen_prueba.jpg")

if CLAVE and ENDPOINT and RUTA.exists():
    datos = RUTA.read_bytes()
    print("Validación:", valida_imagen(datos))
    print()

    d = describe(datos)
    print(f"Descripción: {d['texto']}  (confianza {d['confianza']:.2f})")
    print()

    print("Etiquetas:")
    for t in etiqueta(datos)[:8]:
        print(f"  {t['etiqueta']:22} {t['confianza']:.2f}")

    objetos = detecta_objetos(datos)
    print(f"\nObjetos detectados: {len(objetos)}")
    for o in objetos:
        print(f"  {o['objeto']:22} {o['confianza']:.2f}  {o['recuadro']}")
elif not RUTA.exists():
    print(f"Coloca una imagen en {RUTA} para ejecutar esta celda")

In [ ]:
if CLAVE and ENDPOINT and RUTA.exists() and objetos:
    display(dibuja_recuadros(datos, objetos))

## 6. OCR y por qué el contraste importa

`read` extrae texto impreso y manuscrito, y devuelve la **posición de cada línea** como un
polígono, no como un rectángulo: el texto puede estar girado.

Prueba el OCR con dos fotos del mismo documento, una bien iluminada y otra con poco contraste o a
contraluz. La diferencia en el resultado es la respuesta a por qué la calidad de la captura no es
un detalle del usuario, sino un requisito del sistema.

In [ ]:
RUTA_DOC = pathlib.Path("documento.jpg")

if CLAVE and ENDPOINT and RUTA_DOC.exists():
    ocr = lee_texto(RUTA_DOC.read_bytes())
    print(f"Líneas reconocidas: {len(ocr['lineas'])}\n")
    print(ocr["texto_completo"][:800])
else:
    print(f"Coloca un documento escaneado o fotografiado en {RUTA_DOC}")

## 7. Pedir varias cosas en una llamada

Si necesitas descripción, etiquetas y objetos, **no hagas tres llamadas**. Se factura por llamada.

In [ ]:
def analiza_completo(imagen, idioma="es") -> dict:
    """Descripción, etiquetas, objetos y OCR en una sola petición."""
    datos = _analiza_imagen(imagen, ["caption", "tags", "objects", "read"], idioma)
    return {
        "descripcion": (datos.get("captionResult") or {}).get("text"),
        "etiquetas": [t["name"] for t in (datos.get("tagsResult") or {}).get("values", [])],
        "objetos": len((datos.get("objectsResult") or {}).get("values", [])),
        "lineas_texto": sum(
            len(b.get("lines", []))
            for b in (datos.get("readResult") or {}).get("blocks", [])
        ),
    }


if CLAVE and ENDPOINT and RUTA.exists():
    print(analiza_completo(RUTA.read_bytes()))

No todas las características se pueden combinar en cualquier versión del API, y algunas solo están
en determinadas regiones. Cuando una combinación devuelve 400, la respuesta dice cuál es el
problema: **léela**, no vayas probando.

## Lo que te llevas a `servicios/vision.py`

`valida_imagen`, `_analiza_imagen` y las cuatro funciones públicas. `dibuja_recuadros` **no**: eso
es presentación y va en la interfaz, no en el módulo de servicio.

Esa separación es la que se corrige en la práctica. La regla: si una función necesita Pillow para
dibujar o Streamlit para mostrar, no pertenece a `servicios/`.

Siguiente: **UD2.05**, voz.